# 🔬 Global Patent Intelligence Pipeline
**ETL → Supabase PostgreSQL → Reports & Visualizations (No SQLite)**

```
Google Drive TSVs → Clean chunk-by-chunk → Write CSVs → Load Supabase → SQL Queries → Reports + 8 Visuals → Download ZIP
```

### ⚠️ Before running
- Your 4 TSV files must be in Google Drive at **`MyDrive/raw/`**
  - `g_patent.tsv`
  - `g_inventor_disambiguated.tsv`
  - `g_assignee_disambiguated.tsv`
  
- Rotate your Supabase DB password before entering it below.
- Run cells **one at a time, top to bottom**.

---
## Cell 1 — Install Libraries

In [10]:
!pip install -q pandas sqlalchemy psycopg2-binary matplotlib seaborn
print('✅ Libraries installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 118.0 MB/s eta 0:00:00
✅ Libraries installed


---
## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

---
## Cell 3 — Project Folder Setup

In [ ]:
from pathlib import Path
import os, time, json
import pandas as pd

# ── Raw TSV files location on Google Drive ────────────────────────────────────
RAW_DIR = Path('/content/drive/MyDrive/raw')

# ── Colab working directories ─────────────────────────────────────────────────
BASE        = Path('/content/patent_project')
CLEAN_DIR   = BASE / 'data' / 'cleaned' # Defined here globally
REPORTS_DIR = BASE / 'reports'
VISUALS_DIR = BASE / 'visuals'

for d in [CLEAN_DIR, REPORTS_DIR, VISUALS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Verify raw files exist before doing any work ──────────────────────────────
EXPECTED = [
    'g_patent.tsv',
    'g_inventor_disambiguated.tsv',
    'g_assignee_disambiguated.tsv',
]
missing = [f for f in EXPECTED if not (RAW_DIR / f).exists()]
if missing:
    print('❌ Missing files in MyDrive/raw/:')
    for m in missing:
        print(f'   {m}')
    raise FileNotFoundError('Upload the missing TSV files to Google Drive and re-run.')
else:
    print('✅ All 3 raw TSV files found in Google Drive')
    for f in EXPECTED:
        size = (RAW_DIR / f).stat().st_size / 1_000_000
        print(f'   {f}  ({size:.1f} MB)')

---
## Cell 4 — Cleaning Functions (chunk-by-chunk, writes CSV directly)

In [ ]:
CHUNK_SIZE = 50_000

PATENT_COLUMNS   = ['patent_id', 'patent_title', 'patent_date']
INVENTOR_COLUMNS = ['patent_id', 'inventor_id',
                    'disambig_inventor_name_first', 'disambig_inventor_name_last',
                    'location_id']
ASSIGNEE_COLUMNS = ['patent_id', 'assignee_id',
                    'disambig_assignee_organization', 'assignee_type',
                    'location_id']



def clean_patent_chunk(df):
    df = df.copy().rename(columns={'patent_title': 'title', 'patent_date': 'filing_date'})
    df = df.dropna(subset=['patent_id']).drop_duplicates(subset=['patent_id'])
    df['title'] = df['title'].fillna('Untitled Patent')
    df['filing_date'] = pd.to_datetime(df['filing_date'], errors='coerce')
    df['year'] = df['filing_date'].dt.year.astype('Int64')
    df['filing_date'] = df['filing_date'].dt.strftime('%Y-%m-%d')
    df['filing_date'] = df['filing_date'].where(df['filing_date'].notna(), None)
    df['year'] = df['year'].astype(object).where(df['year'].notna(), None)
    return df[['patent_id', 'title', 'filing_date', 'year']]


def clean_inventor_chunk(df):
    df = df.copy()
    df['name'] = (df['disambig_inventor_name_first'].fillna('') + ' ' +
                  df['disambig_inventor_name_last'].fillna('')).str.strip()
    df['name'] = df['name'].replace('', 'Unknown Inventor')

    df = df[['inventor_id', 'name', 'location_id']]
    return df.dropna(subset=['inventor_id']).drop_duplicates(subset=['inventor_id'])


def clean_patent_inventor_chunk(df):
    return df[['patent_id', 'inventor_id']].copy().dropna().drop_duplicates()


def clean_company_chunk(df):
    df = df.copy()
    df['name'] = df['disambig_assignee_organization'].fillna('Unknown Company')
    df['assignee_type'] = df['assignee_type'].fillna('Unknown')
    # Keep location_id column
    df = df[['assignee_id', 'name', 'location_id', 'assignee_type']]
    return df.dropna(subset=['assignee_id']).drop_duplicates(subset=['assignee_id'])


def clean_patent_company_chunk(df):
    return df[['patent_id', 'assignee_id']].copy().dropna().drop_duplicates()


def stream_and_write(tsv_name, usecols, clean_fn, csv_name, label=None):
    """Read a TSV in chunks, clean each chunk, write directly to CSV."""
    label = label or csv_name
    out = CLEAN_DIR / csv_name
    if out.exists():
        out.unlink()   # always start fresh
    first_write = True
    total_rows = 0
    for chunk in pd.read_csv(
        RAW_DIR / tsv_name, sep='\t', usecols=usecols,
        chunksize=CHUNK_SIZE, dtype=str, on_bad_lines='skip'
    ):
        cleaned = clean_fn(chunk)
        if cleaned.empty:
            continue
        cleaned.to_csv(out, mode='a', index=False, header=first_write)
        first_write = False
        total_rows += len(cleaned)
        print(f'   {label}: {total_rows:,} rows written', end='\r')
    status = '✅' if total_rows else '⚠️ '
    print(f'{status} {label:<30} {total_rows:>10,} rows')
    return total_rows


print('✅ Cleaning functions defined (location_id kept, locations table not used)')

---
## Cell 5 — Run Cleaning Pipeline  ⏳

In [ ]:
t0 = time.time()

# ── Patents ───────────────────────────────────────────────────────────────────
print('🔄 Patents...')
stream_and_write('g_patent.tsv', PATENT_COLUMNS,
                 clean_patent_chunk, 'clean_patents.csv', 'clean_patents.csv')

# ── Inventors + patent_inventor links ─────────────────────────────────────────
print('\n🔄 Inventors...')
stream_and_write('g_inventor_disambiguated.tsv', INVENTOR_COLUMNS,
                 clean_inventor_chunk, 'clean_inventors.csv', 'clean_inventors.csv')

print('🔄 Patent-inventor links...')
stream_and_write('g_inventor_disambiguated.tsv', INVENTOR_COLUMNS,
                 clean_patent_inventor_chunk, 'patent_inventor.csv', 'patent_inventor.csv')

# ── Companies + patent_company links ──────────────────────────────────────────
print('\n🔄 Companies...')
stream_and_write('g_assignee_disambiguated.tsv', ASSIGNEE_COLUMNS,
                 clean_company_chunk, 'clean_companies.csv', 'clean_companies.csv')

print('🔄 Patent-company links...')
stream_and_write('g_assignee_disambiguated.tsv', ASSIGNEE_COLUMNS,
                 clean_patent_company_chunk, 'patent_company.csv', 'patent_company.csv')

# Locations table is NOT processed – we drop it entirely.

elapsed = (time.time() - t0) / 60
print(f'\n✅ Cleaning complete in {elapsed:.1f} min')
print('\nCleaned files:')
for f in sorted(CLEAN_DIR.iterdir()):
    if f.suffix == '.csv':
        size = f.stat().st_size / 1_000_000
        print(f'   {f.name}  ({size:.1f} MB)')

---
## Cell 6 — Connect to Supabase (secure password prompt)

In [2]:
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text
import os, time, json
import pandas as pd

# === Session Pooler (IPv4) – from your dashboard ===
POOLER_HOST = 'aws-1-eu-central-1.pooler.supabase.com'
DB_PORT = 5432
DB_NAME = 'postgres'
DB_USER = 'postgres.xsnuvpagkcqvirfxixai'
DB_PASSWORD = 'Mwesiarnold@2025'

# Encode password to handle special characters like @
encoded_pw = quote_plus(DB_PASSWORD)

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{encoded_pw}@{POOLER_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"

engine = create_engine(DATABASE_URL, pool_pre_ping=True, connect_args={'connect_timeout': 15})

try:
    with engine.connect() as conn:
        result = conn.execute(text('SELECT version()')).fetchone()
        print('✅ Connected to Supabase via Session Pooler (IPv4)')
        print(f'   Version: {result[0][:65]}')
except Exception as e:
    print(f'❌ Connection failed: {e}')
    print('   → Make sure your Supabase project is NOT paused (resume if needed).')

✅ Connected to Supabase via Session Pooler (IPv4)
   Version: PostgreSQL 17.6 on aarch64-unknown-linux-gnu, compiled by gcc (GC


---
## Cell 6a — Database Query Helper Functions
These functions provide robust querying capabilities with retry logic and ensure an extended timeout for long-running operations.

In [3]:
import psycopg2 # Required for specific error handling in run_query
import time
import pandas as pd
from sqlalchemy import text

# --- Set a generous global statement timeout for database queries ---
with engine.connect() as conn:
    conn.execute(text("SET statement_timeout = '1200s'")) # Increased timeout to 1200 seconds (20 minutes)
    print("✅ Database statement timeout set to 1200 seconds (20 minutes)")

def run_query(query, error_msg="", retries=3, delay=5):
    """Executes a SQL query with retry and exponential backoff logic."""
    for i in range(retries):
        try:
            return pd.read_sql(query, engine)
        except Exception as e:
            if isinstance(e, (psycopg2.errors.QueryCanceled,  # Specific to psycopg2 timeout
                              psycopg2.OperationalError)):  # General operational errors
                print(f"  {error_msg}: Attempt {i+1}/{retries} failed due to timeout. Retrying in {delay}s... (Error: {e})")
                time.sleep(delay)
                delay *= 2  # Exponential backoff
            else:
                print(f"{error_msg}: {e}")
                return pd.DataFrame()
    print(f"  {error_msg}: All {retries} attempts failed. Returning empty DataFrame.")
    return pd.DataFrame()

def run_query_chunked(query, chunk_size=10000, error_msg="", retries=3, delay=5):
    """Stream a query in chunks using OFFSET/LIMIT (for very large tables), with retry logic."""
    offset = 0
    total_fetched = 0
    while True:
        paginated_query = f"{query} LIMIT {chunk_size} OFFSET {offset}"
        # Use the run_query function for each chunk to incorporate retry logic
        chunk = run_query(paginated_query, error_msg=f"{error_msg} (chunk offset {offset})", retries=retries, delay=delay)
        if chunk.empty:
            if offset == 0: # If first chunk is empty, there might be no data at all
                break
            # If not first chunk, and chunk is empty after retries, then we're done or failed
            if len(chunk) < chunk_size: # Less than a full chunk means we've reached the end of the data
                break
            else: # If chunk is empty and should not be, means query failed completely
                print(f"  {error_msg} (chunk offset {offset}): Failed to fetch chunk after all retries.")
                break
        yield chunk
        total_fetched += len(chunk)
        offset += chunk_size
        print(f"    Fetched {total_fetched:,} rows...", end='\r')
    print(f"\n✅ Finished fetching {total_fetched:,} rows in chunks.")


✅ Database statement timeout set to 1200 seconds (20 minutes)


---
## Cell 7 — Apply PostgreSQL Schema

In [ ]:
SCHEMA_SQL = """
DROP TABLE IF EXISTS patent_company CASCADE;
DROP TABLE IF EXISTS patent_inventor CASCADE;
DROP TABLE IF EXISTS companies CASCADE;
DROP TABLE IF EXISTS inventors CASCADE;
DROP TABLE IF EXISTS patents CASCADE;

CREATE TABLE patents (
    patent_id   TEXT PRIMARY KEY,
    title       TEXT,
    filing_date DATE,
    year        INTEGER
);

CREATE TABLE inventors (
    inventor_id TEXT PRIMARY KEY,
    name        TEXT,
    location_id TEXT          -- kept for completeness (no foreign key)
);

CREATE TABLE companies (
    assignee_id   TEXT PRIMARY KEY,
    name          TEXT,
    location_id   TEXT,       -- kept for completeness
    assignee_type TEXT
);

CREATE TABLE patent_inventor (
    patent_id   TEXT REFERENCES patents(patent_id),
    inventor_id TEXT REFERENCES inventors(inventor_id),
    PRIMARY KEY (patent_id, inventor_id)
);

CREATE TABLE patent_company (
    patent_id   TEXT REFERENCES patents(patent_id),
    assignee_id TEXT REFERENCES companies(assignee_id),
    PRIMARY KEY (patent_id, assignee_id)
);

CREATE INDEX idx_patents_year   ON patents(year);
CREATE INDEX idx_companies_name ON companies(name);
"""

with engine.begin() as conn:
    for stmt in SCHEMA_SQL.split(';'):
        s = stmt.strip()
        if s:
            conn.exec_driver_sql(s)

print('✅ PostgreSQL schema applied (location_id preserved, no locations table)')

---
## Cell 8 — Load Cleaned CSVs into Supabase  ⏳

In [ ]:
from getpass import getpass
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from sqlalchemy.pool import NullPool
from pathlib import Path
import pandas as pd
import time

# --- 1. Connection ─────────────────────────────────────────────────────────────
POOLER_HOST = 'aws-1-eu-central-1.pooler.supabase.com'
DB_PORT     = 6543
DB_NAME     = 'postgres'
DB_USER     = 'postgres.xsnuvpagkcqvirfxixai'
DB_PASSWORD = getpass('🔑 Supabase password: ')

DATABASE_URL = (
    f'postgresql+psycopg2://{DB_USER}:{quote_plus(DB_PASSWORD)}'
    f'@{POOLER_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require'
)
engine = create_engine(DATABASE_URL, poolclass=NullPool,
                       connect_args={'connect_timeout': 15})

with engine.connect() as conn:
    conn.execute(text('SELECT 1'))
print('✅ Connected to Supabase')

# --- 2. Truncate tables before loading (safe to rerun) ─────────────────────────
# Truncate in reverse FK order, then load in forward FK order
TRUNCATE_ORDER = [
    'patent_company', 'patent_inventor', 'companies', 'inventors', 'patents'
]
with engine.begin() as conn:
    for tbl in TRUNCATE_ORDER:
        conn.execute(text(f'TRUNCATE TABLE {tbl} CASCADE'))
print('✅ Tables cleared — safe to reload')

# --- 3. Load CSVs ──────────────────────────────────────────────────────────────
# CLEAN_DIR is now defined globally in Cell 3

TABLES = [
    ('patents',         'clean_patents.csv',   ['patent_id'],                 {'parse_dates': ['filing_date']}),
    ('inventors',       'clean_inventors.csv', ['inventor_id'],               {}),
    ('companies',       'clean_companies.csv', ['assignee_id'],               {}),
    ('patent_inventor', 'patent_inventor.csv', ['patent_id', 'inventor_id'],  {}),
    ('patent_company',  'patent_company.csv',  ['patent_id', 'assignee_id'],  {}),
]

CHUNK_SIZE = 10_000
t0 = time.time()

for table, file_name, pk_cols, extra_kw in TABLES:
    csv_path = CLEAN_DIR / file_name
    if not csv_path.exists():
        print(f'⚠️  {file_name} not found — skipping {table}')
        continue

    print(f'\n📥 {table}...')
    df = pd.read_csv(csv_path, **extra_kw)
    before = len(df)
    df = df.drop_duplicates(subset=pk_cols)
    print(f'   {before:,} rows → {len(df):,} after dedup')

    total = 0
    for start in range(0, len(df), CHUNK_SIZE):
        chunk = df.iloc[start:start + CHUNK_SIZE]
        chunk.to_sql(table, engine, if_exists='append', index=False, method='multi')
        total += len(chunk)
        print(f'   {total:,} / {len(df):,}', end='\r')
    print(f'   ✅ {total:,} rows loaded')

print(f'\n✅ Done in {(time.time()-t0)/60:.1f} min')

---

---
## Cell 9 — Run All SQL Queries

In [4]:
QUERIES = {

    # Q1 – Patent trend by year
    'patent_trends.csv': """
        SELECT year, COUNT(*) AS patent_count
        FROM patents
        WHERE year IS NOT NULL
        GROUP BY year ORDER BY year
    """,

    # Q2 – Top 20 companies
    'top_companies.csv': """
        SELECT c.name, COUNT(*) AS patent_count
        FROM patent_company pc
        JOIN companies c ON c.assignee_id = pc.assignee_id
        GROUP BY c.name ORDER BY patent_count DESC LIMIT 20
    """,

    # Q3 – Assignee type distribution
    'assignee_types.csv': """
        SELECT assignee_type, COUNT(*) AS company_count
        FROM companies
        GROUP BY assignee_type ORDER BY company_count DESC
    """,

    # Q4 – JOIN: patent + inventor + company sample
    'patent_detail_sample.csv': """
        SELECT p.patent_id, p.title, p.year,
               i.name  AS inventor_name,
               c.name  AS company_name
        FROM patents p
        JOIN patent_inventor pi  ON pi.patent_id  = p.patent_id
        JOIN inventors i         ON i.inventor_id = pi.inventor_id
        LEFT JOIN patent_company pc ON pc.patent_id  = p.patent_id
        LEFT JOIN companies c    ON c.assignee_id = pc.assignee_id
        WHERE p.year IS NOT NULL
        LIMIT 500
    """,
}

results = {}
print('Running queries...')
# Use the run_query function defined in a previous cell (e.g., LzsryppzUCgY or bMXse2kSuzpc)
# Assuming run_query is available in the global scope from previous cell execution
for name, sql in QUERIES.items():
    try:
        df = run_query(sql, error_msg=f"Error fetching {name}")
        if not df.empty:
            results[name] = df
            print(f'  ✅ {name:<30}  {len(results[name]):>8,} rows')
        else:
            print(f'  ❌ {name}: Query returned no data or failed after retries.')
    except Exception as e:
        print(f'  ❌ {name}: An unexpected error occurred: {e}')

print('\n✅ All queries complete')

Running queries...
  ✅ patent_trends.csv                     50 rows
  ✅ top_companies.csv                     20 rows
  ✅ assignee_types.csv                    16 rows
  ✅ patent_detail_sample.csv             500 rows

✅ All queries complete


## Cell 8a — Pre-compute Large Datasets for Visualizations  ⏳

This cell pre-computes two large datasets (`assignee_growth_data` and `collab_df`) by streaming data from Supabase in chunks. This prevents query timeouts that can occur when trying to fetch and aggregate very large tables in a single query. The results are then stored as global DataFrames for use in the dashboard and static visualizations.

In [5]:
import pandas as pd
import time

# ============================================================
# PRE-COMPUTE assignee growth data (chunked, no aggregation in DB)
# ============================================================
print("Pre-computing assignee growth data (streaming)...")
base_assignee_query = """
    SELECT p.year, c.assignee_type
    FROM patents p
    JOIN patent_company pc ON p.patent_id = pc.patent_id
    JOIN companies c ON pc.assignee_id = c.assignee_id
    WHERE p.year IS NOT NULL AND c.assignee_type IS NOT NULL
      AND p.year >= 2000
"""
assignee_chunks = []
# Explicitly setting chunk_size to 5000 for this stream to further reduce load
for chunk in run_query_chunked(base_assignee_query, chunk_size=5000, error_msg="Error fetching assignee growth chunk"):
    assignee_chunks.append(chunk)

if assignee_chunks:
    assignee_raw = pd.concat(assignee_chunks, ignore_index=True)
    # Perform the grouping and counting in Pandas
    assignee_growth_data = assignee_raw.groupby(['assignee_type', 'year']).size().reset_index(name='cnt')
    assignee_growth_data = assignee_growth_data.sort_values(['assignee_type', 'year']).reset_index(drop=True)
    print(f"  ✅ Pre-computed {len(assignee_growth_data):,} aggregated rows of assignee growth data.")
else:
    assignee_growth_data = pd.DataFrame()
    print("  ⚠️ Pre-computation of assignee growth data resulted in an empty DataFrame after fetching.")

# ============================================================
# PRE-COMPUTE collaboration data (streaming raw, then aggregate in Pandas)
# ============================================================
print("Pre-computing collaboration data (streaming raw and aggregating in Pandas)...")
collab_query_raw_data = """
    SELECT i.name as inventor, c.name as company
    FROM patent_inventor pi
    JOIN inventors i ON pi.inventor_id = i.inventor_id
    JOIN patent_company pc ON pi.patent_id = pc.patent_id
    JOIN companies c ON pc.assignee_id = c.assignee_id
    JOIN patents p ON pc.patent_id = p.patent_id
    WHERE p.year IS NOT NULL AND p.year >= 2000
"""
all_collab_data = []
# Using run_query_chunked for raw collaboration data
for chunk in run_query_chunked(collab_query_raw_data, chunk_size=10000, error_msg="Error fetching collaboration raw data chunk"):
    all_collab_data.append(chunk)

if all_collab_data:
    collab_raw_df = pd.concat(all_collab_data, ignore_index=True)
    # Perform aggregation and limit in Pandas
    collab_df = collab_raw_df.groupby(['inventor', 'company']).size().reset_index(name='strength')
    collab_df = collab_df.sort_values(by='strength', ascending=False).head(5000) # Keep top 5000 for manageability
    print(f"  ✅ Pre-computed {len(collab_df)} collaboration pairs.")
else:
    collab_df = pd.DataFrame()
    print("  ⚠️ No collaboration data fetched.")

Pre-computing assignee growth data (streaming)...
Error fetching assignee growth chunk (chunk offset 80000): (psycopg2.errors.QueryCanceled) canceling statement due to statement timeout

[SQL: 
    SELECT p.year, c.assignee_type
    FROM patents p
    JOIN patent_company pc ON p.patent_id = pc.patent_id
    JOIN companies c ON pc.assignee_id = c.assignee_id
    WHERE p.year IS NOT NULL AND c.assignee_type IS NOT NULL
      AND p.year >= 2000
 LIMIT 5000 OFFSET 80000]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

✅ Finished fetching 80,000 rows in chunks.
  ✅ Pre-computed 196 aggregated rows of assignee growth data.
Pre-computing collaboration data (streaming raw and aggregating in Pandas)...

✅ Finished fetching 8,784 rows in chunks.
  ✅ Pre-computed 5000 collaboration pairs.


In [6]:
# ============================================================
# Custom top inventors calculation (to avoid supabaseDB disk full>>free tier)
# ============================================================
from collections import Counter
from sqlalchemy import text

print("Computing top inventors via streaming...")


# Stream patent_inventor and join with inventors
inventor_counter = Counter()
chunk_size = 100000  # rows per fetch

# Load all inventors into a dict (id -> name)
print("Loading inventors dictionary...")
inventors_df = pd.read_sql("SELECT inventor_id, name FROM inventors", engine)
inventor_dict = dict(zip(inventors_df['inventor_id'], inventors_df['name']))
print(f"Loaded {len(inventor_dict):,} inventors")

# Stream patent_inventor in chunks
print("Streaming patent_inventor table...")
offset = 0
total_processed = 0
while True:
    query = f"""
        SELECT patent_id, inventor_id
        FROM patent_inventor
        LIMIT {chunk_size} OFFSET {offset}
    """
    chunk = pd.read_sql(query, engine)
    if chunk.empty:
        break
    for inv_id in chunk['inventor_id']:
        inventor_counter[inv_id] += 1
    total_processed += len(chunk)
    print(f"  Processed {total_processed:,} rows", end='\r')
    offset += chunk_size

print(f"\nProcessed {total_processed:,} rows. Aggregating names...")
# Convert inventor_id counts to name counts
name_counter = Counter()
for inv_id, count in inventor_counter.items():
    name = inventor_dict.get(inv_id, 'Unknown')
    name_counter[name] += count

# Get top 20
top_inventors = name_counter.most_common(20)
top_inventors_df = pd.DataFrame(top_inventors, columns=['name', 'patent_count'])
print("Top inventors computed successfully")

# Store in results dictionary (overwrite the usual query)
results['top_inventors.csv'] = top_inventors_df
print("✅ Stored top_inventors.csv in results")

Computing top inventors via streaming...
Loading inventors dictionary...
Loaded 4,294,034 inventors
Streaming patent_inventor table...

Processed 575,781 rows. Aggregating names...
Top inventors computed successfully
✅ Stored top_inventors.csv in results


---
## Cell 10 — Console Report (Terminal Output)

In [7]:
# Summary query – includes all required counts
summary_sql = """
    SELECT
        (SELECT COUNT(*) FROM patents)         AS total_patents,
        (SELECT COUNT(*) FROM inventors)       AS total_inventors,
        (SELECT COUNT(*) FROM companies)       AS total_companies,
        (SELECT COUNT(*) FROM patent_inventor) AS inventor_links,
        (SELECT COUNT(*) FROM patent_company)  AS company_links
"""
summary = pd.read_sql(summary_sql, engine).iloc[0].to_dict()

W = 62
print('=' * W)
print('             GLOBAL PATENT INTELLIGENCE REPORT')
print('=' * W)
print(f"  Total Patents         : {int(summary.get('total_patents', 0)):>12,}")
print(f"  Total Inventors       : {int(summary.get('total_inventors', 0)):>12,}")
print(f"  Total Companies       : {int(summary.get('total_companies', 0)):>12,}")
print(f"  Patent-Inventor Links : {int(summary.get('inventor_links', 0)):>12,}")
print(f"  Patent-Company Links  : {int(summary.get('company_links', 0)):>12,}")

print('\n' + '-' * W)
print('    TOP 10 COMPANIES BY PATENT COUNT')
print('-' * W)
if 'top_companies.csv' in results:
    for i, row in results['top_companies.csv'].head(10).reset_index(drop=True).iterrows():
        print(f"  {i+1:>2}. {str(row['name'])[:44]:<44}  {int(row['patent_count']):>8,}")
else:
    print("    Top companies data not available")

print('\n' + '-' * W)
print('    TOP 10 INVENTORS')
print('-' * W)
if 'top_inventors.csv' in results:
    for i, row in results['top_inventors.csv'].head(10).reset_index(drop=True).iterrows():
        print(f"  {i+1:>2}. {str(row['name'])[:44]:<44}  {int(row['patent_count']):>8,}")
else:
    print("    Top inventors data not available – run custom aggregation first")

print('\n' + '=' * W)
print('  Source: Supabase PostgreSQL — PatentsView Disambiguated Data')
print('=' * W)

             GLOBAL PATENT INTELLIGENCE REPORT
  Total Patents         :    9,454,161
  Total Inventors       :    4,294,034
  Total Companies       :      572,495
  Patent-Inventor Links :      575,781
  Patent-Company Links  :      166,716

--------------------------------------------------------------
    TOP 10 COMPANIES BY PATENT COUNT
--------------------------------------------------------------
   1. SAMSUNG DISPLAY CO., LTD.                        3,381
   2. International Business Machines Corporation      3,054
   3. CANON KABUSHIKI KAISHA                           1,732
   4. Unknown Company                                  1,701
   5. SONY GROUP CORPORATION                           1,262
   6. Fujitsu Limited                                  1,020
   7. Kabushiki Kaisha Toshiba                           988
   8. Intel Corporation                                  954
   9. General Electric Company                           941
  10. HITACHI, LTD.                          

---
## Cell 11 — Save CSV + JSON Reports

In [8]:
from pathlib import Path
import pandas as pd
import json   # <-- add this import

BASE = Path('/content/patent_project')
REPORTS_DIR = BASE / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# If 'summary' is not defined, re-fetch it (optional, but safe)
if 'summary' not in dir():
    summary_sql = """
        SELECT
            (SELECT COUNT(*) FROM patents)         AS total_patents,
            (SELECT COUNT(*) FROM inventors)       AS total_inventors,
            (SELECT COUNT(*) FROM companies)       AS total_companies,
            (SELECT COUNT(*) FROM patent_inventor) AS inventor_links,
            (SELECT COUNT(*) FROM patent_company)  AS company_links
    """
    summary = pd.read_sql(summary_sql, engine).iloc[0].to_dict()
    print("✅ Summary re-fetched")

csv_exports = [
    ('patent_trends.csv',           'patent_trends.csv'),
    ('top_companies.csv',           'top_companies.csv'),
    ('top_inventors.csv',           'top_inventors.csv'),
    ('assignee_types.csv',          'assignee_types.csv'),
    ('patent_detail_sample.csv',    'patent_detail_sample.csv'),
]

for key, fname in csv_exports:
    if key in results:
        results[key].to_csv(REPORTS_DIR / fname, index=False)
        print(f'  💾 {fname}')
    else:
        print(f'  ⚠️ {key} not found in results — skipping')

# JSON report
json_report = {
    'summary': {k: int(v) for k, v in summary.items()},
    'top_companies': results['top_companies.csv'].head(20).to_dict(orient='records'),
    'top_inventors': results['top_inventors.csv'].head(20).to_dict(orient='records'),
    'patent_trends': results['patent_trends.csv'].to_dict(orient='records'),
    'assignee_types': results['assignee_types.csv'].to_dict(orient='records'),
}

(REPORTS_DIR / 'patent_report.json').write_text(
    json.dumps(json_report, indent=2, default=str)
)
print('  💾 patent_report.json')
print('\n✅ Reports saved')

  💾 patent_trends.csv
  💾 top_companies.csv
  💾 top_inventors.csv
  💾 assignee_types.csv
  💾 patent_detail_sample.csv
  💾 patent_report.json

✅ Reports saved


---
## GRADIO DASHBOARD

In [9]:
!pip install gradio plotly pandas python-dotenv
import gradio as gr
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import psycopg2
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import os
from datetime import datetime

In [10]:
# ============================================================
# PRE‑COMPUTE TECHNOLOGY CATEGORY COUNTS (run once)
# ============================================================
import pandas as pd
import re
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from pathlib import Path

# --- Connect to Supabase (same credentials) ---
POOLER_HOST = 'aws-1-eu-central-1.pooler.supabase.com'
DB_PORT = 6543
DB_NAME = 'postgres'
DB_USER = 'postgres.xsnuvpagkcqvirfxixai'
DB_PASSWORD = 'Mwesiarnold@2025'

encoded_pw = quote_plus(DB_PASSWORD)
DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{encoded_pw}@{POOLER_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
engine = create_engine(DATABASE_URL, pool_pre_ping=True)

# --- Refined keywords (more specific to reduce ambiguity) ---
keywords = {
    'AI/ML': r'\b(artificial intelligence|machine learning|neural network|deep learning|natural language processing|computer vision|large language model|generative ai|llm)\b',
    'Battery/Energy': r'\b(lithium ion|battery|energy storage|fuel cell|solar cell|photovoltaic|wind turbine|renewable energy|electric vehicle)\b',
    'Telecom/5G': r'\b(5g|6g|wireless communication|cellular network|signal processing|antenna array|mimo|beamforming|telecommunications)\b',
    'Biotech/Pharma': r'\b(gene editing|crispr|therapeutic|vaccine|antibody|personalized medicine|mrna|biologics|protein|gene)\b',
    'Semiconductor': r'\b(semiconductor|integrated circuit|chip|transistor|cmos|wafer|microchip|processor|memory device)\b',
    'Software/Cloud': r'\b(cloud computing|blockchain|distributed system|virtualization|container|kubernetes|microservice|software as a service)\b'
}

# --- Process all patents in chunks ---
chunk_size = 100000  # adjust if memory is tight
category_counts = {cat: 0 for cat in keywords}
category_counts['Other'] = 0

print("Processing all patent titles (this may take a few minutes)...")
total_processed = 0

for chunk in pd.read_sql("SELECT title FROM patents WHERE title IS NOT NULL", engine, chunksize=chunk_size):
    for title in chunk['title']:
        t = str(title).lower()
        matched = False
        for cat, pattern in keywords.items():
            if re.search(pattern, t):
                category_counts[cat] += 1
                matched = True
                break
        if not matched:
            category_counts['Other'] += 1
    total_processed += len(chunk)
    print(f"Processed {total_processed:,} titles...", end='\r')

print(f"\n✅ Done! Processed {total_processed:,} patents.")
print("Category counts:")
for cat, count in category_counts.items():
    print(f"  {cat}: {count:,}")

# --- Save to CSV (so dashboard can read it) ---
DATA_DIR = Path('/content/patent_project/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
output_path = DATA_DIR / 'technology_categories.csv'
df_cats = pd.DataFrame(category_counts.items(), columns=['category', 'count'])
df_cats.to_csv(output_path, index=False)
print(f"\nSaved to {output_path}")

Processing all patent titles (this may take a few minutes)...
Processed 9,454,161 titles...
✅ Done! Processed 9,454,161 patents.
Category counts:
  AI/ML: 23,003
  Battery/Energy: 108,187
  Telecom/5G: 58,874
  Biotech/Pharma: 52,601
  Semiconductor: 338,542
  Software/Cloud: 80,074
  Other: 8,792,880

Saved to /content/patent_project/data/technology_categories.csv


In [19]:
import gradio as gr
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import re
from pathlib import Path
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import time
import psycopg2

# ============================================================
# 1. Supabase connection
#    (Assuming engine is globally available from Cell 6)
# ============================================================
POOLER_HOST = 'aws-1-eu-central-1.pooler.supabase.com'
DB_PORT = 6543
DB_NAME = 'postgres'
DB_USER = 'postgres.xsnuvpagkcqvirfxixai'
DB_PASSWORD = 'Mwesiarnold@2025'

encoded_pw = quote_plus(DB_PASSWORD)
DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{encoded_pw}@{POOLER_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
engine = create_engine(DATABASE_URL, pool_pre_ping=True, connect_args={'connect_timeout': 15})


# ============================================================
# 2. All plot functions
# ============================================================
def plot_trend():
    df = run_query("SELECT year, COUNT(*) as cnt FROM patents WHERE year IS NOT NULL GROUP BY year ORDER BY year")
    if df.empty:
        return go.Figure().add_annotation(text="No data", xref="paper", yref="paper", showarrow=False)
    return px.line(df, x='year', y='cnt', title='1. Patent Filings Over Time', markers=True)

def plot_top_companies():
    df = run_query("""
        SELECT c.name, COUNT(*) as cnt
        FROM patent_company pc
        JOIN companies c ON pc.assignee_id = c.assignee_id
        GROUP BY c.name ORDER BY cnt DESC LIMIT 20
    """)
    if df.empty:
        return go.Figure().add_annotation(text="No company data", xref="paper", yref="paper", showarrow=False)
    return px.bar(df, y='name', x='cnt', title='2. Top 20 Companies', orientation='h')

def plot_top_inventors():
    try:
        df_ids = run_query("""
            SELECT inventor_id, COUNT(*) as patent_count
            FROM patent_inventor
            GROUP BY inventor_id
            ORDER BY patent_count DESC
            LIMIT 20
        """)
        if df_ids.empty:
            return go.Figure().add_annotation(text="No inventor data", xref="paper", yref="paper", showarrow=False)

        inventor_ids = tuple(df_ids['inventor_id'])
        if len(inventor_ids) == 1:
            in_clause = f"('{inventor_ids[0]}')"
        else:
            in_clause = str(inventor_ids)

        df_names = run_query(f"""
            SELECT inventor_id, name
            FROM inventors
            WHERE inventor_id IN {in_clause}
        """)
        if df_names.empty:
            return go.Figure().add_annotation(text="No inventor names found", xref="paper", yref="paper", showarrow=False)

        df = df_ids.merge(df_names, on='inventor_id')
        df = df.sort_values('patent_count', ascending=True)

        fig = px.bar(df, y='name', x='patent_count', title='3. Top 20 Inventors', orientation='h')
        fig.update_layout(xaxis_title="Patent Count", yaxis_title="Inventor")
        return fig
    except Exception as e:
        print(f"Error in plot_top_inventors: {e}")
        return go.Figure().add_annotation(text=f"Error: {str(e)[:100]}", xref="paper", yref="paper", showarrow=False)

def plot_assignee_type():
    df = run_query("SELECT assignee_type, COUNT(*) as cnt FROM companies GROUP BY assignee_type")
    if df.empty:
        return go.Figure().add_annotation(text="No assignee data", xref="paper", yref="paper", showarrow=False)
    mapping = {
        '1': "1: Unassigned", '2': "2: US Company", '3': "3: Foreign Company",
        '4': "4: US Individual", '5': "5: Foreign Individual", '6': "6: US Federal Gov",
        '7': "7: Foreign Gov", '8': "8: US County Gov", '9': "9: US State Gov"
    }
    df['label'] = df['assignee_type'].astype(str).map(mapping).fillna("Other (" + df['assignee_type'].astype(str) + ")")
    fig = px.bar(df, y='label', x='cnt', title='4. Assignee Type Distribution', orientation='h')
    fig.update_layout(xaxis_title="Number of Companies", yaxis_title="Assignee Type")
    return fig

def plot_yoy():
    df = run_query("SELECT year, COUNT(*) as cnt FROM patents WHERE year IS NOT NULL GROUP BY year ORDER BY year")
    if df.empty or len(df) < 2:
        return go.Figure().add_annotation(text="Insufficient data", xref="paper", yref="paper", showarrow=False)
    df['yoy'] = df['cnt'].pct_change() * 100
    return px.bar(df, x='year', y='yoy', title='5. Year-over-Year Growth (%)')

def plot_decade():
    df = run_query("SELECT year FROM patents WHERE year IS NOT NULL")
    if df.empty:
        return go.Figure().add_annotation(text="No year data", xref="paper", yref="paper", showarrow=False)
    df['decade'] = (df['year'] // 10) * 10
    decade_counts = df.groupby('decade').size().reset_index(name='count')
    return px.line(decade_counts, x='decade', y='count', title='6. Patents by Decade', markers=True)

def plot_companies_over_time():
    top5 = run_query("""
        SELECT c.name
        FROM patent_company pc
        JOIN companies c ON pc.assignee_id = c.assignee_id
        GROUP BY c.name ORDER BY COUNT(*) DESC LIMIT 5
    """)
    if top5.empty:
        return go.Figure().add_annotation(text="No company data", xref="paper", yref="paper", showarrow=False)
    top5_names = tuple(top5['name'].tolist())
    df = run_query(f"""
        SELECT c.name, p.year, COUNT(*) as cnt
        FROM patent_company pc
        JOIN companies c ON pc.assignee_id = c.assignee_id
        JOIN patents p ON pc.patent_id = p.patent_id
        WHERE c.name IN {top5_names} AND p.year IS NOT NULL
        GROUP BY c.name, p.year
    """)
    if df.empty:
        return go.Figure().add_annotation(text="No time series data", xref="paper", yref="paper", showarrow=False)
    pivot = df.pivot(index='name', columns='year', values='cnt').fillna(0)
    return px.imshow(pivot, title='7. Patent Counts for Top 5 Companies Over Time', aspect="auto")

def plot_growth_by_assignee():
    global assignee_growth_data # Access globally pre-computed data
    if assignee_growth_data.empty:
        return go.Figure().add_annotation(text="No assignee growth data", xref="paper", yref="paper", showarrow=False)

    df = assignee_growth_data.copy()
    mapping = {
        '1': "Unassigned", '2': "US Company", '3': "Foreign Company",
        '4': "US Individual", '5': "5: Foreign Individual", '6': "6: US Federal Gov",
        '7': "7: Foreign Gov", '8': "8: US County Gov", '9': "9: US State Gov"
    }
    df['type_label'] = df['assignee_type'].astype(str).map(mapping).fillna("Other (" + df['assignee_type'].astype(str) + ")")
    fig = px.line(df, x='year', y='cnt', color='type_label',
                  title='8. Patent Growth by Assignee Type',
                  labels={'cnt': 'Patent Count', 'year': 'Year', 'type_label': 'Assignee Type'})
    fig.update_layout(legend_title_text='Assignee Type')
    return fig

def plot_inventor_productivity():
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    try:
        df = run_query("SELECT COUNT(*) AS patent_count FROM patent_inventor GROUP BY inventor_id")
        if df.empty:
            return go.Figure().add_annotation(text="No productivity data", xref="paper", yref="paper", showarrow=False)

        fig_hist = px.histogram(df, x='patent_count', nbins=50, log_y=True)
        fig_hist.update_layout(xaxis_title="Patents per inventor", yaxis_title="Number of inventors (log scale)")
        median_val = df['patent_count'].median()
        fig_hist.add_vline(x=median_val, line_dash="dash", line_color="red",
                           annotation_text=f"Median = {median_val:.0f}", annotation_position="top")

        fig_box = px.box(df, y='patent_count')
        fig_box.update_yaxes(type="log", title="Patents per inventor (log scale)")

        cdf_vals = df['patent_count'].value_counts().sort_index().reset_index()
        cdf_vals.columns = ['patent_count', 'inventor_count']
        cdf_vals['cumulative_inventors'] = cdf_vals['inventor_count'].cumsum()
        total_inventors = cdf_vals['cumulative_inventors'].iloc[-1]
        cdf_vals['cdf'] = cdf_vals['cumulative_inventors'] / total_inventors

        fig_cdf = px.line(cdf_vals, x='patent_count', y='cdf', markers=True)
        fig_cdf.update_layout(xaxis_title="Patents per inventor", yaxis_title="Cumulative proportion of inventors",
                              yaxis_tickformat=".0%")
        fig_cdf.add_hline(y=0.5, line_dash="dash", line_color="gray", opacity=0.5)
        fig_cdf.add_hline(y=0.8, line_dash="dash", line_color="gray", opacity=0.5)

        fig = make_subplots(rows=3, cols=1,
                            subplot_titles=("Histogram (log y)", "Boxplot (log y)", "Cumulative Distribution"),
                            vertical_spacing=0.1)
        for trace in fig_hist.data:
            fig.add_trace(trace, row=1, col=1)
        for trace in fig_box.data:
            fig.add_trace(trace, row=2, col=1)
        for trace in fig_cdf.data:
            fig.add_trace(trace, row=3, col=1)

        fig.update_xaxes(title_text="Patents per inventor", row=1, col=1)
        fig.update_yaxes(title_text="Number of inventors (log)", row=1, col=1, type="log")
        fig.update_xaxes(title_text="", row=2, col=1)
        fig.update_yaxes(title_text="Patents per inventor (log)", row=2, col=1, type="log")
        fig.update_xaxes(title_text="Patents per inventor", row=3, col=1)
        fig.update_yaxes(title_text="Cumulative proportion", row=3, col=1, tickformat=".0%")
        fig.update_layout(height=1000, showlegend=False, title_text="9. Inventor Productivity Analysis")
        return fig
    except Exception as e:
        print(f"Error in plot_inventor_productivity: {e}")
        return go.Figure().add_annotation(text=f"Error: {str(e)[:100]}", xref="paper", yref="paper", showarrow=False)

def plot_technology_mix():
    try:
        csv_path = '/content/patent_project/data/technology_categories.csv'
        if not Path(csv_path).exists():
            return go.Figure().add_annotation(
                text="Technology categories not pre\u2011computed. Run the pre\u2011computation cell first.",
                xref="paper", yref="paper", showarrow=False)
        df = pd.read_csv(csv_path)
        df = df.sort_values('count', ascending=True)
        fig = px.bar(df, y='category', x='count', title='10. Technology Focus Areas (based on full patent titles)', orientation='h')
        fig.update_layout(xaxis_title="Number of Patents", yaxis_title="Technology Area")
        return fig
    except Exception as e:
        print(f"Error in plot_technology_mix: {e}")
        return go.Figure().add_annotation(text="Error loading tech data", xref="paper", yref="paper", showarrow=False)

def plot_collaboration():
    global collab_df # Access globally pre-computed data
    try:
        # Ensure collab_df is accessible (it's pre-computed globally now)
        global collab_df
        if collab_df.empty:
            return go.Figure().add_annotation(text="No collaboration data pre-computed", xref="paper", yref="paper", showarrow=False)

        # Get top 20 inventors and top 20 companies by total collaboration strength from the pre-computed data
        top_inventors = collab_df.groupby('inventor')['strength'].sum().nlargest(20).index.tolist()
        top_companies = collab_df.groupby('company')['strength'].sum().nlargest(20).index.tolist()

        # Filter collaboration data for these top entities
        df_filt = collab_df[
            (collab_df['inventor'].isin(top_inventors)) &
            (collab_df['company'].isin(top_companies))
        ]

        if df_filt.empty:
            return go.Figure().add_annotation(text="No collaboration between selected top inventors and top companies", xref="paper", yref="paper", showarrow=False)

        pivot = df_filt.pivot_table(index='inventor', columns='company', values='strength').fillna(0)

        fig = px.imshow(pivot, color_continuous_scale='Viridis',
                        title='11. Top 20 Inventor-Company Collaboration Strength (Patents together)',
                        labels=dict(color="Patents Together"))
        fig.update_layout(
            xaxis_title="Company",
            yaxis_title="Inventor",
            xaxis={'tickangle': 45},
            height=800,
            width=1000
        )
        return fig
    except Exception as e:
        print(f"Error in plot_collaboration: {e}")
        return go.Figure().add_annotation(text=f"Error: {str(e)[:100]}", xref="paper", yref="paper", showarrow=False)

def plot_top_inventors_for_top_companies():
    try:
        # Get the top 5 companies by total patent count using a more efficient query
        top_companies_df = run_query("""
            SELECT c.name
            FROM companies c
            JOIN (
                SELECT assignee_id, COUNT(*) as patent_count
                FROM patent_company
                GROUP BY assignee_id
                ORDER BY patent_count DESC
                LIMIT 5
            ) AS top_assignees ON c.assignee_id = top_assignees.assignee_id
            ORDER BY top_assignees.patent_count DESC
        """, error_msg="Error fetching top companies for affiliation")

        if top_companies_df.empty:
            return go.Figure().add_annotation(text="No top companies data available.", xref="paper", yref="paper", showarrow=False)

        top_company_names_list = top_companies_df['name'].tolist()
        if not top_company_names_list:
            return go.Figure().add_annotation(text="No top companies names extracted.", xref="paper", yref="paper", showarrow=False)

        # Convert to tuple for SQL IN clause
        # Handle single element tuple correctly by adding a comma (e.g., ('Company A',))
        if len(top_company_names_list) == 1:
            top_company_names = f"('{top_company_names_list[0]}')"
        else:
            top_company_names = tuple(top_company_names_list)

        # Query the top inventors associated with these top 5 companies, by total patents
        query = f"""
            SELECT i.name as inventor_name, c.name as company_name, COUNT(DISTINCT p.patent_id) as patent_count
            FROM patents p
            JOIN patent_company pc ON p.patent_id = pc.patent_id
            JOIN companies c ON pc.assignee_id = c.assignee_id
            JOIN patent_inventor pi ON p.patent_id = pi.patent_id
            JOIN inventors i ON pi.inventor_id = i.inventor_id
            WHERE c.name IN {top_company_names}
            GROUP BY i.name, c.name
            ORDER BY patent_count DESC
            LIMIT 25 -- Fetch top 25 distinct inventor-company links
        """
        df = run_query(query, error_msg="Error fetching top inventors by company affiliation")
        if df.empty:
            return go.Figure().add_annotation(text="No inventor-company affiliation data available.", xref="paper", yref="paper", showarrow=False)

        # Combine inventor name and company for unique labels if needed, or use color
        df['inventor_company'] = df['inventor_name'] + ' (' + df['company_name'].apply(lambda x: x.split(' ')[0]) + ')'

        fig = px.bar(df.sort_values(by='patent_count', ascending=True),
                     y='inventor_company',
                     x='patent_count',
                     color='company_name',
                     title='12. Top Inventors Associated with Top 5 Companies',
                     labels={'patent_count': 'Number of Patents', 'inventor_company': 'Inventor (Company)', 'company_name': 'Company'},
                     orientation='h')
        fig.update_layout(showlegend=True, height=700)
        return fig
    except Exception as e:
        print(f"Error in plot_top_inventors_for_top_companies: {e}")
        return go.Figure().add_annotation(text=f"Error: {str(e)[:100]}", xref="paper", yref="paper", showarrow=False)


# ============================================================
# 3. Dashboard interface (with enriched historical insights)
# ============================================================
with gr.Blocks(title="Patent Intelligence Dashboard") as demo:
    gr.Markdown("# 🔬 Global Patent Intelligence Dashboard")
    gr.Markdown("**Descriptive & Diagnostic Analytics** \u2013 Based on PatentsView data (1976\u20132020)")

    with gr.Tabs():
        # ---------- Tab 1: Trends & Growth ----------
        with gr.TabItem("📊 Trends & Growth"):
            with gr.Row():
                with gr.Column():
                    plot1 = gr.Plot()
                with gr.Column():
                    plot5 = gr.Plot()
            with gr.Row():
                plot6 = gr.Plot()
            demo.load(plot_trend, outputs=plot1)
            demo.load(plot_yoy, outputs=plot5)
            demo.load(plot_decade, outputs=plot6)

            gr.Markdown("""
            ### 📈 Historical Context & Insights: Trends & Growth

            **Patent Filings Over Time**
            - Patent activity was low before the 20th century. The sharp rise from the 1970s reflects the information age (computers, telecom).
            - The **Great Depression (1930s)** and **World War II** caused declines \u2013 but our data starts at 1976, so we see only modern growth.
            - The **COVID\u201319 pandemic (2020)** may show a dip or a surge in health\u2013related patents; check the year\u2013over\u2013year graph.

            **Year\u2013over\u2013Year Growth**
            - Positive bars = expansion (e.g., .com boom of the 1990s, AI explosion after 2010).
            - Negative bars = slowdowns (e.g., 2008 financial crisis, possibly 2020 lockdowns).

            **Patents by Decade**
            - Each bar shows total patents in that decade. The 1990s and 2000s saw massive growth due to the Internet, mobile phones, and later AI.
            - If the 2010s are still rising, we are in a golden age of innovation (AI, biotech, renewable energy).
            """)

        # ---------- Tab 2: Companies & Assignees ----------
        with gr.TabItem("🏢 Companies & Assignees"):
            with gr.Row():
                plot2 = gr.Plot()
            with gr.Row():
                plot4 = gr.Plot()
            with gr.Row():
                plot8 = gr.Plot()
            demo.load(plot_top_companies, outputs=plot2)
            demo.load(plot_assignee_type, outputs=plot4)
            demo.load(plot_growth_by_assignee, outputs=plot8)

            gr.Markdown("""
            ### 💰 Historical Context & Insights: Companies & Assignees

            **Top 20 Companies**
            - Leaders like IBM, Samsung, Canon have shaped the digital age. IBM dominated computing patents from the 1970s\u20131990s; Samsung rose later in semiconductors and displays.
            - The presence of \u251CUnknown Company\u251D suggests incomplete data (patents without an assignee).

            **Assignee Type Distribution**
            - **US companies** usually dominate, reflecting strong corporate R&D.
            - **Foreign companies** indicate globalisation of innovation (e.g., Japanese and Korean firms).
            - **Individuals** (inventors filing without a company) \u2013 a small fraction, but historically many great inventions started in garages.

            **Growth by Assignee Type**
            - Compare the curves: Are US companies growing faster than foreign firms? Has government patenting increased (e.g., due to climate or health crises)?
            """)

        # ---------- Tab 3: Inventors ----------
        with gr.TabItem("🧑\u200d🔬 Inventors"):
            with gr.Row():
                plot3 = gr.Plot()
            with gr.Row():
                plot9 = gr.Plot()
            demo.load(plot_top_inventors, outputs=plot3)
            demo.load(plot_inventor_productivity, outputs=plot9)

            gr.Markdown("""
            ### 🧠 Historical Context & Insights: Inventors

            **Top 20 Inventors**
            - These individuals often work for the leading companies. Their high patent counts show they are key to their firm\u2019s innovation engine.

            **Inventor Productivity Distribution**
            - **Extreme skew**: Most inventors file only one patent. This is the **Pareto principle (80/20 rule)** \u2013 a small number of inventors produce most patents.
            - The **median = 1** confirms that more than half of all inventors have a single patent.
            - The **boxplot** shows outliers (the top 1% of inventors).
            - The **cumulative curve** tells you, e.g., \u251Cthe top 10% of inventors hold 60% of all patents\u251D \u2013 you can read exact values from the plot.
            """)

        # ---------- Tab 4: Technology & Collaboration ----------
        with gr.TabItem("🔗 Technology & Collaboration"):
            with gr.Row():
                plot10 = gr.Plot()
            with gr.Row():
                plot7 = gr.Plot()
            with gr.Row():
                plot11 = gr.Plot()
            with gr.Row():
                plot12 = gr.Plot() # New plot for top inventors per company
            demo.load(plot_technology_mix, outputs=plot10)
            demo.load(plot_companies_over_time, outputs=plot7)
            demo.load(plot_collaboration, outputs=plot11)
            demo.load(plot_top_inventors_for_top_companies, outputs=plot12) # Load the new plot

            gr.Markdown("""
            ### 🤝 Historical Context & Insights: Technology & Collaboration

            **Technology Focus Areas**
            - **AI/ML**: Despite being a hot topic only since 2010, patents in AI go back decades (early neural networks in the 1980s). The high count today reflects the current boom.
            - **Biotech/Pharma**: Obtaining a drug patent is very difficult (requires clinical trials, safety proof). Therefore the relatively lower count is not a sign of less innovation \u2013 it reflects high regulatory barriers.
            - **Semiconductor and Telecom**: Mature fields with steady output.
            - **\u251COther\u251D** is large because many patents do not contain our keywords \u2013 for example, mechanical engineering, chemistry, or materials science.

            **Top 5 Companies Over Time (Heatmap)**
            - See how innovation leadership shifted: IBM dominated in the 1980s\u20131990s; Samsung Display and Sony have risen in the 2000s\u20132010s.
            - Darker cells = more patents. This visualisation instantly shows who was leading in which decade.

            **Inventor\u2013Company Collaboration**
            - The heatmap shows which top inventors work closely with which top companies. Darker = more joint patents.

            **Top Inventors Associated with Top 5 Companies**
            - This chart highlights the most prolific inventors affiliated with the leading patent-holding companies, providing insight into the individual contributors driving innovation within these firms.
            """)

# Launch
demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://065b674c417f5cbbb9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Cell 12 — 8 Visualizations 📊

In [12]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd
import numpy as np
import re
from pathlib import Path
from sqlalchemy import create_engine, text
import psycopg2 # Import psycopg2 for specific error handling

# --- Paths ---
BASE = Path('/content/patent_project')
VISUALS_DIR = BASE / 'visuals'
VISUALS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
TAB10 = sns.color_palette('tab10', 20)

def save(name):
    plt.savefig(VISUALS_DIR / name, dpi=150, bbox_inches='tight')
    print(f'  💾 {name}')
    plt.close()

def fmt_k(ax, axis='x'):
    f = mticker.FuncFormatter(lambda x, _: f'{x:,.0f}')
    (ax.xaxis if axis == 'x' else ax.yaxis).set_major_formatter(f)

# --- Supabase connection and global functions (run_query, run_query_chunked) are defined in prior cells ---
# The 'engine' object is assumed to be globally available from cell RydHCNsBUCgV
# The `run_query` and `run_query_chunked` functions are defined in cell 248e2989

# ============================================================
# Pre-computed assignee growth data and collaboration data are available globally
# ============================================================

# ============================================================
# Fetch other base data (small queries)
# ============================================================
print("Fetching other base data...")
patent_trends = run_query("SELECT year, COUNT(*) as patent_count FROM patents WHERE year IS NOT NULL GROUP BY year ORDER BY year", error_msg="Error fetching patent trends")
top_companies = run_query("SELECT c.name, COUNT(*) as patent_count FROM patent_company pc JOIN companies c ON pc.assignee_id = c.assignee_id GROUP BY c.name ORDER BY patent_count DESC LIMIT 20", error_msg="Error fetching top companies")
top_inventors = run_query("SELECT i.name, COUNT(*) as patent_count FROM patent_inventor pi JOIN inventors i ON pi.inventor_id = i.inventor_id GROUP BY i.name ORDER BY patent_count DESC LIMIT 20", error_msg="Error fetching top inventors")
assignee_types = run_query("SELECT assignee_type, COUNT(*) as company_count FROM companies GROUP BY assignee_type", error_msg="Error fetching assignee types")
df_year = run_query("SELECT year FROM patents WHERE year IS NOT NULL", error_msg="Error fetching years for patents")
df_inv_prod = run_query("SELECT COUNT(*) as patent_count FROM patent_inventor GROUP BY inventor_id", error_msg="Error fetching inventor productivity data")
tech_df = run_query("SELECT title FROM patents WHERE title IS NOT NULL LIMIT 100000", error_msg="Error fetching tech titles")
top5 = run_query("""
    SELECT c.name, p.year, COUNT(*) as cnt
    FROM patent_company pc
    JOIN companies c ON pc.assignee_id = c.assignee_id
    JOIN patents p ON pc.patent_id = p.patent_id
    WHERE c.name IN (
        SELECT c2.name FROM patent_company pc2 JOIN companies c2 ON pc2.assignee_id = c2.assignee_id
        GROUP BY c2.name ORDER BY COUNT(*) DESC LIMIT 5
    )
    AND p.year IS NOT NULL AND p.year >= 2000
    GROUP BY c.name, p.year
""", error_msg="Error fetching top 5 companies over time")

# ============================================================
# 1. Patent Filings Over Time
# ============================================================
if not patent_trends.empty:
    df = patent_trends
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(df['year'], df['patent_count'], lw=2.5, color='steelblue', marker='o', ms=3)
    ax.fill_between(df['year'], df['patent_count'], alpha=0.15, color='steelblue')
    ax.set_title('Global Patent Filings by Year', fontsize=16, fontweight='bold')
    ax.set_xlabel('Year'); ax.set_ylabel('Number of Patents')
    fmt_k(ax, 'y')
    save('v1_patent_trends.png')

# 2. Top 20 Companies
if not top_companies.empty:
    df = top_companies.head(20).sort_values('patent_count')
    fig, ax = plt.subplots(figsize=(12, 9))
    bars = ax.barh(df['name'], df['patent_count'], color=TAB10)
    for b in bars:
        ax.text(b.get_width() + df['patent_count'].max() * 0.005,
                b.get_y() + b.get_height()/2,
                f"{b.get_width():,.0f}", va='center', fontsize=8)
    ax.set_title('Top 20 Companies by Patent Count', fontsize=15, fontweight='bold')
    ax.set_xlabel('Patent Count')
    fmt_k(ax, 'x')
    save('v2_top_companies.png')

# 3. Top 20 Inventors
if not top_inventors.empty:
    df = top_inventors.head(20).sort_values('patent_count')
    fig, ax = plt.subplots(figsize=(12, 9))
    ax.barh(df['name'], df['patent_count'], color=sns.color_palette('flare', len(df)))
    ax.set_title('Top 20 Inventors by Patent Count', fontsize=15, fontweight='bold')
    ax.set_xlabel('Patent Count')
    fmt_k(ax, 'x')
    save('v3_top_inventors.png')

# 4. Assignee Type Distribution
if not assignee_types.empty:
    df = assignee_types.copy()
    mapping = {'1':'Unassigned','2':'US Company','3':'Foreign Company','4':'US Individual',
               '5':'Foreign Individual','6':'US Federal Gov','7':'Foreign Gov',
               '8':'US County Gov','9':'US State Gov'}
    df['label'] = df['assignee_type'].astype(str).map(mapping).fillna('Other')
    df = df.sort_values('company_count', ascending=False)
    fig, ax = plt.subplots(figsize=(10, max(6, len(df)*0.4)))
    bars = ax.barh(df['label'], df['company_count'], color=sns.color_palette('Set2', len(df)))
    ax.set_title('Number of Companies by Assignee Type', fontsize=15, fontweight='bold')
    ax.set_xlabel('Number of Companies')
    for bar in bars:
        ax.text(bar.get_width() + max(df['company_count'])*0.01,
                bar.get_y() + bar.get_height()/2,
                f'{int(bar.get_width()):,}', va='center', fontsize=9)
    fmt_k(ax, 'x')
    save('v4_assignee_types.png')

# 5. Year-over-Year Growth
if not patent_trends.empty:
    df = patent_trends.copy()
    df['yoy_growth'] = df['patent_count'].pct_change() * 100
    df = df.dropna()
    bar_colors = ['crimson' if v < 0 else 'seagreen' for v in df['yoy_growth']]
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(df['year'], df['yoy_growth'], color=bar_colors, width=0.8)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title('Year-over-Year Patent Filing Growth Rate (%)', fontsize=15, fontweight='bold')
    ax.set_xlabel('Year'); ax.set_ylabel('Growth (%)')
    save('v5_yoy_growth.png')

# 6. Patents by Decade
if not df_year.empty:
    df_year['decade'] = (df_year['year'] // 10) * 10
    decade_counts = df_year.groupby('decade').size().reset_index(name='count')
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(decade_counts['decade'], decade_counts['count'], marker='o', lw=2.5, color='steelblue')
    ax.fill_between(decade_counts['decade'], decade_counts['count'], alpha=0.15, color='steelblue')
    ax.set_title('Patents by Decade', fontsize=15, fontweight='bold')
    ax.set_xlabel('Decade'); ax.set_ylabel('Number of Patents')
    fmt_k(ax, 'y')
    save('v6_patents_by_decade.png')

# 7. Top 5 Companies Over Time Heatmap
if not top5.empty:
    pivot = top5.pivot(index='name', columns='year', values='cnt').fillna(0)
    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(pivot, aspect='auto', cmap='YlOrRd')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title('Patent Counts for Top 5 Companies Over Time (2000\u2013present)', fontsize=15, fontweight='bold')
    plt.colorbar(im, ax=ax, label='Patent Count')
    save('v7_top5_companies_heatmap.png')
else:
    print("  ⚠️ No data for top 5 companies heatmap")

# 8. Growth by Assignee Type (using pre\u2011computed DataFrame)
# Access globally pre-computed assignee_growth_data
global assignee_growth_data
if not assignee_growth_data.empty:
    map_type = {'1':'Unassigned','2':'US Company','3':'Foreign Company','4':'US Individual',
                '5':'Foreign Individual','6':'US Fed Gov','7':'Foreign Gov','8':'US County Gov','9':'US State Gov'}
    assignee_growth_data['type_label'] = assignee_growth_data['assignee_type'].astype(str).map(map_type).fillna('Other')
    fig, ax = plt.subplots(figsize=(13, 6))
    for label, group in assignee_growth_data.groupby('type_label'):
        ax.plot(group['year'], group['cnt'], marker='o', label=label, lw=2)
    ax.set_title('Patent Growth by Assignee Type (2000\u2013present)', fontsize=15, fontweight='bold')
    ax.set_xlabel('Year'); ax.set_ylabel('Patent Count')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    fmt_k(ax, 'y')
    save('v8_growth_by_assignee.png')
else:
    print("  ⚠️ No assignee growth data (pre\u2011computation failed)")

# 9. Inventor Productivity Distribution
if not df_inv_prod.empty:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=False)
    ax1.hist(df_inv_prod['patent_count'], bins=50, edgecolor='black', log=True)
    ax1.set_title('Histogram of Patents per Inventor (log y)')
    ax1.set_xlabel('Patents per inventor')
    ax1.set_ylabel('Number of inventors (log)')
    ax2.boxplot(df_inv_prod['patent_count'], vert=False, showfliers=True)
    ax2.set_title('Boxplot (log scale)')
    ax2.set_xlabel('Patents per inventor (log)')
    ax2.set_xscale('log')
    plt.tight_layout()
    save('v9_inventor_productivity.png')
else:
    print("  ⚠️ No inventor productivity data")

# 10. Technology Focus Areas
if not tech_df.empty:
    keywords = {
        'AI/ML': r'\b(AI|artificial intelligence|machine learning|neural|deep learning)\b',
        'Battery/Energy': r'\b(battery|energy storage|fuel cell|solar|wind|renewable)\b',
        'Telecom/5G': r'\b(5G|telecom|communication|wireless|signal|antenna)\b',
        'Biotech/Pharma': r'\b(gene|protein|therapeutic|vaccine|cancer|antibody|CRISPR)\b',
        'Semiconductor': r'\b(chip|semiconductor|transistor|integrated circuit|CMOS)\b',
        'Software/Cloud': r'\b(software|cloud computing|blockchain|algorithm|virtual)\b',
    }
    def categorize(title):
        t = str(title).lower()
        for cat, pat in keywords.items():
            if re.search(pat, t, re.IGNORECASE):
                return cat
        return 'Other'
    tech_df['category'] = tech_df['title'].apply(categorize)
    counts = tech_df['category'].value_counts().reset_index()
    counts.columns = ['category', 'count']
    counts = counts.sort_values('count', ascending=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(counts['category'], counts['count'], color='teal')
    ax.set_title('Technology Focus Areas (sample of 100k titles)', fontsize=15, fontweight='bold')
    ax.set_xlabel('Number of Patents')
    fmt_k(ax, 'x')
    save('v10_technology_focus.png')
else:
    print("  ⚠️ No title data for technology focus")

# 11. Collaboration Heatmap (using pre\u2011fetched df)
# Access globally pre-computed collab_df
global collab_df
if not collab_df.empty:
    # Use top 20 inventors and companies by total strength as requested
    top_inv = collab_df.groupby('inventor')['strength'].sum().nlargest(20).index.tolist()
    top_comp = collab_df.groupby('company')['strength'].sum().nlargest(20).index.tolist()
    df_filt = collab_df[collab_df['inventor'].isin(top_inv) & collab_df['company'].isin(top_comp)]
    if not df_filt.empty:
        pivot = df_filt.pivot_table(index='inventor', columns='company', values='strength').fillna(0)
        fig, ax = plt.subplots(figsize=(16, 12)) # Adjusted figsize for better display of top 20x20
        im = ax.imshow(pivot, cmap='Blues', aspect='auto')
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns, rotation=90, ha='right', fontsize=8) # Rotate labels, smaller font
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index, fontsize=8)
        ax.set_title('Top 20 Inventor-Company Collaboration Strength (Patents together, 2000\u2013present)', fontsize=15, fontweight='bold')
        plt.colorbar(im, ax=ax, label='Patents together')
        plt.tight_layout() # Adjust layout to prevent labels from being cut off
    else:
        print("  ⚠️ No filtered collaboration data for heatmap")
else:
    print("  ⚠️ No collaboration data (pre\u2011computation failed)")
save('v11_collaboration_heatmap.png')

print('\n✅ All 11 visualizations processed')


Fetching other base data...
  💾 v1_patent_trends.png
  💾 v2_top_companies.png
  💾 v3_top_inventors.png
  💾 v4_assignee_types.png
  💾 v5_yoy_growth.png
  💾 v6_patents_by_decade.png
  💾 v7_top5_companies_heatmap.png
  💾 v8_growth_by_assignee.png
  💾 v9_inventor_productivity.png
  💾 v10_technology_focus.png
  💾 v11_collaboration_heatmap.png

✅ All 11 visualizations processed


---
## Cell 13 — Download Everything as a ZIP

In [14]:
import zipfile
from google.colab import files as colab_files
from pathlib import Path

ZIP_PATH = '/content/patent_outputs.zip'

# Define necessary directory paths to ensure they are available
BASE = Path('/content/patent_project')
CLEAN_DIR = BASE / 'data' / 'cleaned'
REPORTS_DIR = BASE / 'reports'
VISUALS_DIR = BASE / 'visuals'

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    if REPORTS_DIR.exists():
        for f in sorted(REPORTS_DIR.iterdir()):
            zf.write(f, f'reports/{f.name}')
    if VISUALS_DIR.exists():
        for f in sorted(VISUALS_DIR.iterdir()):
            zf.write(f, f'visuals/{f.name}')
    if CLEAN_DIR.exists():
        for f in sorted(CLEAN_DIR.iterdir()):
            if f.suffix == '.csv': # Only zip CSVs from cleaned_data
                zf.write(f, f'cleaned_data/{f.name}')

size_mb = Path(ZIP_PATH).stat().st_size / 1_000_000
print(f'📦 patent_outputs.zip  ({size_mb:.1f} MB)')
print('   Contents:')
with zipfile.ZipFile(ZIP_PATH) as zf:
    for name in sorted(zf.namelist()):
        print(f'   {name}')

colab_files.download(ZIP_PATH)
print('\n✅ Download started!')

📦 patent_outputs.zip  (0.9 MB)
   Contents:
   reports/assignee_types.csv
   reports/patent_detail_sample.csv
   reports/patent_report.json
   reports/patent_trends.csv
   reports/top_companies.csv
   reports/top_inventors.csv
   visuals/v10_technology_focus.png
   visuals/v11_collaboration_heatmap.png
   visuals/v1_patent_trends.png
   visuals/v2_top_companies.png
   visuals/v3_top_inventors.png
   visuals/v4_assignee_types.png
   visuals/v5_yoy_growth.png
   visuals/v6_patents_by_decade.png
   visuals/v7_top5_companies_heatmap.png
   visuals/v8_growth_by_assignee.png
   visuals/v9_inventor_productivity.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download started!


---
## Cell 14 — (Optional) Push to GitHub

Fill in your credentials, then run.

In [21]:
# ============================================================
# GITHUB PUSH CELL – FIXED & WORKING VERSION
# ============================================================
import os
import shutil
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# === YOUR GITHUB CREDENTIALS ===
GITHUB_USER  = 'AJmight'
GITHUB_REPO  = 'patent_assignment'
GITHUB_EMAIL = 'arnoldmugahi@gmail.com'
GITHUB_TOKEN = "removed for security"

# --- Paths ---
REPORTS_DIR = Path('/content/patent_project/reports')
VISUALS_DIR = Path('/content/patent_project/visuals')
DELIVER = Path('/content/deliverables')
REPO_DIR = Path('/content/patent_repo')

# --- 1. Clean deliverables ---
if DELIVER.exists():
    shutil.rmtree(DELIVER)
DELIVER.mkdir(parents=True, exist_ok=True)

print("📦 Copying deliverables...")

# --- 2. Copy reports ---
if REPORTS_DIR.exists():
    for f in REPORTS_DIR.iterdir():
        if f.suffix in ['.csv', '.json'] and f.is_file():
            shutil.copy(f, DELIVER / f.name)
            print(f"  ✅ {f.name}")
else:
    print("  ⚠️ No reports folder found")

# --- 3. Copy visuals ---
if VISUALS_DIR.exists():
    for f in VISUALS_DIR.iterdir():
        if f.suffix == '.png' and f.is_file():
            shutil.copy(f, DELIVER / f.name)
            print(f"  ✅ {f.name}")
else:
    print("  ⚠️ No visuals folder found")

# --- 4. Copy notebook ---
NOTEBOOK_SOURCE = Path('/content/drive/MyDrive/Colab Notebooks/patent_pipeline (1).ipynb')
NOTEBOOK_DEST = DELIVER / 'patent_pipeline.ipynb'

if NOTEBOOK_SOURCE.exists():
    shutil.copy(NOTEBOOK_SOURCE, NOTEBOOK_DEST)
    print("  ✅ Notebook copied from Drive")
else:
    fallback = Path('/content/patent_pipeline.ipynb')
    if fallback.exists():
        shutil.copy(fallback, NOTEBOOK_DEST)
        print("  ✅ Fallback notebook copied")
    else:
        print("  ⚠️ No notebook found")

# --- 5. requirements.txt ---
(DELIVER / 'requirements.txt').write_text("""pandas
sqlalchemy
psycopg2-binary
matplotlib
seaborn
plotly
gradio
python-dotenv
kaleido""")

# --- 6. README (FULL DETAILED GUIDE) ---
(DELIVER / 'README.md').write_text("""
 # Patent Intelligence Pipeline

## 📌 Project Overview
This project builds a complete data pipeline for analyzing global patent trends using real USPTO data. It extracts raw TSV files, cleans and transforms them, loads the data into a cloud database (Supabase), runs analytical SQL queries, generates reports, and creates interactive visualizations.

##  Download Large Data Files
The cleaned CSV files (`clean_patents.csv`, `clean_inventors.csv`, `clean_companies.csv`) are **too large for GitHub** (each >100MB).
🔗 **Download them from Google Drive:**
[**Click here to download the complete zip file (1.2 GB)**](https://drive.google.com/file/d/19_mGCeohI5vRYf-esB5KEOHBfIPpcTCb/view?usp=drive_link)
*(Replace this link with your actual shared folder link)*

## 🌐 View the Interactive Dashboard
The dashboard is built with **Gradio** and can be launched in two ways:

### Option 1: Run the notebook in Colab
IF THIS LINK ISNT ACTIVE {https://065b674c417f5cbbb9.gradio.live/}
1. Open the file `patent_pipeline.ipynb` in Google Colab.
2. Run from cell Cell 6 — Connect to Supabase (secure password prompt) to GRADIO DASHBOARD Excluding cell 7 and 8 . The gradio cell will create a **public Gradio link** (e.g., `https://xxxx.gradio.live`).
3. Click that link to explore the 11 interactive charts with live database queries.

### Option 2: Static reports (already in this repo)
- The `reports/` folder contains CSV/JSON outputs.
- The `visuals/` folder contains all 11 PNG charts.

##  Technology Stack
- **Python** (ETL, analysis, dashboard)
- **Pandas** (data cleaning)
- **SQLAlchemy** (database connection)
- **Supabase (PostgreSQL)** (cloud database – free tier)
- **Plotly / Matplotlib / Seaborn** (static and interactive charts)
- **Gradio** (interactive dashboard)
- **Google Colab** (execution environment)

##  Repository Contents
- `patent_pipeline.ipynb` – main Colab notebook with all steps
- `requirements.txt` – Python dependencies
- `reports/` – CSV and JSON outputs (top companies, inventors, trends, etc.)
- `visuals/` – static PNG charts (11 visualizations)

##  How to Reproduce (Step-by-Step)

### Prerequisites
- A **Google account** (to use Colab and Google Drive)
- A **Supabase account** (free tier – create a project at [supabase.com](https://supabase.com))
- Raw patent TSV files from the USPTO PatentsView bulk data API:
  - `g_patent.tsv`
  - `g_inventor_disambiguated.tsv`
  - `g_assignee_disambiguated.tsv`
  (Place these in a folder on your Google Drive, e.g., `MyDrive/patent_data/`)

### Step 1: Open the Notebook in Colab
- Click the `patent_pipeline.ipynb` file in this repository.
- Open it directly in Google Colab (or download and upload to your Drive).

### Step 2: Set Up the Environment
- In the first cell, install dependencies:
  pip install -r requirements.txt
...""")

# --- 7. .gitignore ---
(DELIVER / '.gitignore').write_text("""*.tsv
*.csv
*.zip
__pycache__/
.ipynb_checkpoints
.env""")

# --- 8. Prepare repo ---
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
REPO_DIR.mkdir()

# Copy deliverables into repo
for f in DELIVER.iterdir():
    shutil.copy(f, REPO_DIR / f.name)

# --- 9. GIT SETUP ---
os.chdir(REPO_DIR)

os.system(f'git config --global user.email "{GITHUB_EMAIL}"')
os.system(f'git config --global user.name "{GITHUB_USER}"')

os.system('git init')
os.system('git add .')
os.system('git commit -m "Auto push from Colab"')

# Force branch = main
os.system('git branch -M main')

# --- 🔥 CRITICAL FIX PART ---
print("\n🚀 Pushing to GitHub...")

# Use token DURING push (this is what makes it work)
push_cmd = f'git push https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git main --force'

result = os.system(push_cmd)

if result == 0:
    print("✅ SUCCESSFULLY PUSHED 🎉")
    print(f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}")
else:
    print("❌ Push failed — check token or repo name")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📦 Copying deliverables...
  ✅ assignee_types.csv
  ✅ patent_detail_sample.csv
  ✅ top_inventors.csv
  ✅ top_companies.csv
  ✅ patent_report.json
  ✅ patent_trends.csv
  ✅ v5_yoy_growth.png
  ✅ v7_top5_companies_heatmap.png
  ✅ v3_top_inventors.png
  ✅ v9_inventor_productivity.png
  ✅ v6_patents_by_decade.png
  ✅ v2_top_companies.png
  ✅ v10_technology_focus.png
  ✅ v4_assignee_types.png
  ✅ v8_growth_by_assignee.png
  ✅ v1_patent_trends.png
  ✅ v11_collaboration_heatmap.png
  ⚠️ No notebook found

🚀 Pushing to GitHub...
✅ SUCCESSFULLY PUSHED 🎉
https://github.com/AJmight/patent_assignment
